In [2]:
import os, io, base64, requests, numpy as np
import torch, torch.nn as nn
import torchvision.transforms as T
from PIL import Image

BASE_URL = os.getenv("BASE_URL", "http://154.57.164.66:37670")  # export BASE_URL=

# ---------- model (inlined) ----------
class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(); self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(); self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64*8*8, 128); self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.5); self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu3(self.fc1(x)))
        return self.fc2(x)

# ---------- codecs ----------
def tensor_from_b64_png(b64):
    img = Image.open(io.BytesIO(base64.b64decode(b64)))
    if img.size != (32, 32): raise ValueError("Expected 32x32 PNG")
    return T.ToTensor()(img)                      # (3,32,32) in [0,1]

def b64_png_from_tensor(t):
    arr = (t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

# ---------- fetch weights + challenge ----------
open("cifar10_model_best.pth","wb").write(requests.get(f"{BASE_URL}/model/weights", timeout=30).content)
ck = torch.load("cifar10_model_best.pth", map_location="cpu")
sd = ck["model_state_dict"] if isinstance(ck, dict) and "model_state_dict" in ck else ck
model = CIFAR10CNN().eval(); model.load_state_dict(sd)

ch   = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x    = tensor_from_b64_png(ch["image"])
orig = int(ch["original_class"])
thr  = float(ch["l2_threshold"])                  # 3.5, normalized space
overshoot = float(ch.get("overshoot_hint", 0.02))
max_it    = int(ch.get("max_iterations_hint", 50))
mean = torch.tensor(ch["normalization"]["mean"]).view(3,1,1)
std  = torch.tensor(ch["normalization"]["std"]).view(3,1,1)

to_norm = lambda img01: (img01 - mean) / std
to_01   = lambda imgN : (imgN * std + mean)

# ---------- untargeted L2-PGD in NORMALIZED space (push well past the boundary) ----------
def l2_pgd_away(x01, orig_label, radius, steps=200, step=0.15):
    x0n = to_norm(x01).clone()
    advn = x0n.clone()
    y = torch.tensor([orig_label])
    for _ in range(steps):
        xv = advn.clone().detach().requires_grad_(True)
        logits = model(xv.unsqueeze(0))
        loss = nn.functional.cross_entropy(logits, y)   # MAXIMIZE loss on 'horse'
        g = torch.autograd.grad(loss, xv)[0].flatten()
        with torch.no_grad():
            g = g / (g.norm() + 1e-12)
            advn = advn + (step * g).view_as(advn)      # ascend -> away from horse
            d = (advn - x0n).flatten()
            n = d.norm()
            if n > radius:                               # project onto L2 ball
                advn = x0n + (d * (radius / n)).view_as(advn)
            advn = to_norm(to_01(advn).clamp(0, 1))      # keep [0,1] valid
    return to_01(advn).clamp(0, 1).detach()

# grow the radius until it stays misclassified AFTER the PNG round-trip
best = None
for radius in [1.5, 2.0, 2.5, 3.0, 3.3]:
    adv = l2_pgd_away(x, orig, radius=radius, steps=200, step=0.15)
    b64 = b64_png_from_tensor(adv)
    adv_rt = tensor_from_b64_png(b64)
    p   = pred01(adv_rt)
    l2n = float(torch.norm((to_norm(adv_rt) - to_norm(x)).flatten()))
    print(f"radius={radius:.1f}  pred={p} (orig {orig})  l2_norm={l2n:.3f} (<= {thr})")
    if p != orig and l2n <= thr:
        best = b64; print("  -> flipped & within budget, submitting"); break

if best:
    r = requests.post(f"{BASE_URL}/submit", json={"image": best}, timeout=10)
    print(r.status_code, r.text)
else:
    print("still not flipped — raise step or radius")

# ---------- verify against PNG round-trip in NORMALIZED L2 ----------
b64    = b64_png_from_tensor(adv)
adv_rt = tensor_from_b64_png(b64)
l2n    = float(torch.norm((to_norm(adv_rt) - to_norm(x)).flatten()))
print(f"after PNG -> pred={pred01(adv_rt)} (orig {orig})  l2_norm={l2n:.4f} (<= {thr})")

# ---------- submit ----------
r = requests.post(f"{BASE_URL}/submit", json={"image": b64}, timeout=10)
print(r.status_code, r.text)

radius=1.5  pred=3 (orig 7)  l2_norm=1.519 (<= 3.5)
  -> flipped & within budget, submitting
200 {"flag":"[REDACTED]","message":"Congratulations! Challenge solved!","success":true,"validation":{"adversarial_class":"cat","adversarial_prediction":3,"attack_direction":true,"direction_alignment":0.4523825943470001,"l2_norm":1.5191755294799805,"l2_satisfied":true,"l2_threshold":3.5,"linf_norm":0.039215683937072754,"misclassification":true,"original_class":"horse","original_prediction":7,"reasonable_linf":true,"valid_range":true}}

after PNG -> pred=3 (orig 7)  l2_norm=1.5192 (<= 3.5)
200 {"flag":"[REDACTED]","message":"Congratulations! Challenge solved!","success":true,"validation":{"adversarial_class":"cat","adversarial_prediction":3,"attack_direction":true,"direction_alignment":0.4523825943470001,"l2_norm":1.5191755294799805,"l2_satisfied":true,"l2_threshold":3.5,"linf_norm":0.039215683937072754,"misclassification":true,"original_class":"horse","original_prediction":7,"reasonable_linf":tr